# Healthcare Claims — Visual Analytics
### A Data Science Showcase for Medicaid Cost Analysis

---

**Who is the user?**  
A **Medicaid managed care organization (MCO)** — a health insurer contracted by a state to manage Medicaid benefits. The MCO receives a fixed monthly payment per member (PMPM) and must deliver care within budget. Understanding *where* costs come from is essential to staying solvent and improving outcomes.

**Why does cost analysis matter?**  
Medicaid programs serve high-need, high-cost populations. A small group of members typically drives the majority of spend. Identifying cost patterns early allows the MCO to:
- Target care management resources at the right members
- Detect billing anomalies before they compound
- Report accurately to CMS and state agencies

**This notebook answers 3 questions:**
1. How is cost distributed across members? *(Cost Concentration)*
2. What types of claims drive the most spend? *(Cost Drivers)*
3. Who are the high-cost members and what do they look like? *(Risk Segmentation)*

In [ ]:
import sys, os
os.system(f'"{sys.executable}" -m pip install matplotlib seaborn -q')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120})

# Load data
df = pd.read_csv('../data/sample_claims.csv')
df['service_date'] = pd.to_datetime(df['service_date'])

# Synthetic age groups (representative of Medicaid enrollment mix)
rng = np.random.default_rng(42)
df['age_group'] = rng.choice(['18–34', '35–49', '50–64', '65+'], size=len(df),
                              p=[0.15, 0.25, 0.35, 0.25])

paid = df[df['claim_status'] == 'PAID'].copy()

print(f'  Members:      {df["member_id"].nunique()}')
print(f'  Total claims: {len(df)}')
print(f'  Total paid:   ${paid["paid_amount"].sum():,.0f}')
print(f'  Denial rate:  {(df["claim_status"]=="DENIED").mean():.0%}')

---
## Insight 1: Cost Concentration
**A small group of members drives most of the spend**

- The top 20% of members account for the majority of total paid costs
- Spend distribution is highly skewed — most members cost little, a few cost a lot

👉 **Business implication:** Care management resources (nurse case managers, disease management programs) should be concentrated on the top-cost tier — *not* spread equally across all members. Identifying these members early in the year is the highest-ROI intervention available to an MCO.

In [ ]:
member_spend = (
    paid.groupby('member_id')['paid_amount']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
)
member_spend['cum_pct'] = member_spend['paid_amount'].cumsum() / member_spend['paid_amount'].sum() * 100

cutoff      = max(1, int(len(member_spend) * 0.2))
top20_pct   = member_spend.iloc[:cutoff]['paid_amount'].sum() / member_spend['paid_amount'].sum() * 100
member_pcts = np.linspace(0, 100, len(member_spend))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Pareto curve
ax = axes[0]
ax.plot(member_pcts, member_spend['cum_pct'], color='steelblue', linewidth=2.5)
ax.fill_between(member_pcts, member_spend['cum_pct'], alpha=0.12, color='steelblue')
ax.axvline(20, color='#e74c3c', linestyle='--', linewidth=1.5)
ax.axhline(top20_pct, color='#e67e22', linestyle='--', linewidth=1.5)
ax.annotate(f'Top 20% of members\n= {top20_pct:.0f}% of spend',
            xy=(20, top20_pct), xytext=(35, top20_pct - 18),
            fontsize=10, color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b'))
ax.set_title('Cost Concentration — Pareto Curve', fontweight='bold')
ax.set_xlabel('% of Members (high cost → low cost)')
ax.set_ylabel('Cumulative % of Total Spend')
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)

# Right: bar chart by member (red = top 20%)
ax2 = axes[1]
bar_colors = ['#e74c3c' if i < cutoff else '#aec6cf' for i in range(len(member_spend))]
ax2.bar(range(len(member_spend)), member_spend['paid_amount'], color=bar_colors, width=0.8)
ax2.set_title('Individual Member Spend\n(Red = Top 20%)', fontweight='bold')
ax2.set_xlabel('Members (ranked by spend)')
ax2.set_ylabel('Total Paid ($)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.set_xticks([])

plt.suptitle(f'Total Program Spend: ${paid["paid_amount"].sum():,.0f}  |  '
             f'Top 20% of Members = {top20_pct:.0f}% of Spend',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## Insight 2: Cost Drivers
**Inpatient claims are the dominant cost category by a wide margin**

- Inpatient (IP) claims average 10–20× the cost of outpatient visits, even at lower volume
- Hospital provider type drives nearly all high-cost claims

👉 **Business implication:** Reducing unnecessary inpatient admissions through prior authorization, transitional care programs, and outpatient alternatives yields the largest cost savings. This is where any MCO cost containment strategy should start.

*How this helps decision-making: Program directors use this breakdown to negotiate provider contracts and set medical management priorities for the plan year.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Total spend by claim type
spend_type = paid.groupby('claim_type')['paid_amount'].sum().sort_values(ascending=False)
total_spend = spend_type.sum()
colors = sns.color_palette('Set2', len(spend_type))

bars = axes[0].bar(spend_type.index, spend_type.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, spend_type.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total_spend * 0.01,
                 f'${val:,.0f}\n({val/total_spend:.0%})',
                 ha='center', fontsize=10, fontweight='bold', color='#2c3e50')
axes[0].set_title('Total Paid by Claim Type', fontweight='bold')
axes[0].set_ylabel('Total Paid ($)')
axes[0].set_xlabel('Claim Type  (IP=Inpatient, OP=Outpatient, LTC=Long-term Care, RX=Pharmacy)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_ylim(0, spend_type.max() * 1.3)

# Right: Average cost per claim by type (volume vs unit cost)
avg_type = paid.groupby('claim_type')['paid_amount'].mean().sort_values(ascending=False)
cnt_type = paid.groupby('claim_type')['paid_amount'].count()

bars2 = axes[1].bar(avg_type.index, avg_type.values,
                    color=sns.color_palette('muted', len(avg_type)), edgecolor='white', linewidth=1.5)
for bar, val, ct in zip(bars2, avg_type.values, cnt_type[avg_type.index]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + avg_type.max() * 0.01,
                 f'${val:,.0f}\n({ct} claims)',
                 ha='center', fontsize=10, fontweight='bold', color='#2c3e50')
axes[1].set_title('Average Cost per Claim by Type', fontweight='bold')
axes[1].set_ylabel('Average Paid ($)')
axes[1].set_xlabel('Claim Type')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_ylim(0, avg_type.max() * 1.3)

plt.suptitle('Inpatient Claims: Highest Total Spend AND Highest Unit Cost', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Insight 3: Risk Segmentation
**High-cost members are older and use inpatient care at much higher rates**

- Members split at the median into High-Cost vs Low-Cost groups show distinct age and claim-type patterns
- Inpatient utilization is the clearest separator between the two segments

👉 **Business implication:** Age and inpatient history are the two strongest early signals for identifying rising-risk members. In a production model, these would be inputs to a predictive risk score used to trigger care management outreach *before* a costly event occurs.

*How this helps decision-making: Case managers use segment profiles to prioritize outreach lists. Actuaries use them to validate capitation rates.*

In [ ]:
# Segment members at median spend
member_total = paid.groupby('member_id')['paid_amount'].sum()
high_ids = member_total[member_total >= member_total.median()].index
paid['segment'] = paid['member_id'].isin(high_ids).map({True: 'High-Cost', False: 'Low-Cost'})

seg_summary = (paid.groupby('segment')['paid_amount']
                   .agg(members='count', avg_paid='mean', total_paid='sum'))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = {'High-Cost': '#e74c3c', 'Low-Cost': '#3498db'}

# 1. Average paid per claim by segment
avg_seg = paid.groupby('segment')['paid_amount'].mean()
bars = axes[0].bar(avg_seg.index, avg_seg.values,
                   color=[palette[s] for s in avg_seg.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, avg_seg.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + avg_seg.max() * 0.02,
                 f'${val:,.0f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Avg Claim Cost by Segment', fontweight='bold')
axes[0].set_ylabel('Avg Paid ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_ylim(0, avg_seg.max() * 1.3)

# 2. Claim type distribution by segment
ct_seg = (paid.groupby(['segment', 'claim_type'])
              .size()
              .unstack(fill_value=0)
              .apply(lambda r: r / r.sum() * 100, axis=1))
ct_seg.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', ct_seg.shape[1]),
            edgecolor='white', linewidth=0.8)
axes[1].set_title('Claim Type Mix by Segment\n(% of claims)', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('% of Claims')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Claim Type', fontsize=9)

# 3. Age group distribution by segment
age_order = ['18–34', '35–49', '50–64', '65+']
ag_seg = (paid.groupby(['segment', 'age_group'])
              .size()
              .unstack(fill_value=0)
              .reindex(columns=[c for c in age_order if c in paid['age_group'].unique()]))
ag_seg_pct = ag_seg.apply(lambda r: r / r.sum() * 100, axis=1)
ag_seg_pct.plot(kind='bar', ax=axes[2], color=sns.color_palette('Blues_d', ag_seg_pct.shape[1]),
                edgecolor='white', linewidth=0.8)
axes[2].set_title('Age Group Mix by Segment\n(% of claims)', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('% of Claims')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Age Group', fontsize=9)

plt.suptitle('High-Cost vs Low-Cost Segments — Distinct Profiles Enable Targeted Interventions',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('Segment comparison:')
print(f"  High-Cost avg paid: ${paid[paid['segment']=='High-Cost']['paid_amount'].mean():,.0f}")
print(f"  Low-Cost avg paid:  ${paid[paid['segment']=='Low-Cost']['paid_amount'].mean():,.0f}")

---
## Key Takeaways

- **Cost is highly concentrated.** A small percentage of members drives the majority of total spend — consistent with what Medicaid MCOs see across all markets. Care management programs that identify these members early generate the highest return on investment.

- **Inpatient claims are the primary cost lever.** IP claims represent a fraction of claim volume but dominate total spend. Any cost containment strategy should prioritize reducing avoidable inpatient admissions through prior authorization, transitional care, and chronic disease management.

- **High-cost members have a distinct profile.** Older age groups and higher inpatient utilization clearly distinguish high-cost from low-cost members. These signals are actionable: they can be used to build a risk stratification model that flags rising-risk members *before* a hospitalization occurs.

---

**How a healthcare organization acts on this:**  
A Medicaid MCO would use these insights to build a risk score for each member at enrollment. Members in the top tier receive proactive outreach from a nurse case manager — a proven intervention that reduces inpatient admissions and, by extension, program cost. The SQL analytics layer (notebook 02) provides the underlying queries that feed these scores into a care management workflow system.